# 05 · Improve the verifier, then let an optimizer use it

*Part 3, sections 3-5 and 7.* An optimizer is only as good as its verifier: give it a weak or
gameable judge and it optimizes for the wrong thing. So the order is

1. measure how often the `quality` judge agrees with people, and **align** it if it drifted,
2. only then let an optimizer write a candidate against it,
3. and gate the candidate on the **holdout**, which the optimizer never saw.

In [ ]:
import _nb as nb

experiment_id = nb.setup()

In [ ]:
import mlflow
from selfimprove import config, gate, instructions, signals, store, traces
from selfimprove.workflows import align as align_job

if not config.settings.has_llm:
    print("This notebook needs judge credentials (ANTHROPIC_API_KEY in .env). Cells below will skip.")

## 1. Judge-vs-human agreement on a held-out slice

The judge writes under the same assessment name as the reviewers, so the comparison is direct. The
fit/check split is by trace id, so the slice used to measure is never the slice used to align.

In [ ]:
labeled = [t for t in traces.production() if traces.human_feedback(t) is not None]
fit, check = align_job.split_traces(labeled)
print(len(labeled), "human-labeled traces:", len(fit), "to align on,", len(check), "to measure on")

judge = signals.quality_judge()
if judge is not None:
    before = align_job.agreement(judge, check)
    print(f"the quality judge agrees with people on {before:.0%} of held-out labels")

**Under the hood:** `signals.quality_judge()` first tries `mlflow.genai.scorers.get_scorer(name=
"quality")`, the registered (possibly aligned) judge, and falls back to `make_judge(name="quality", ...)`.
`align_job.agreement()` calls the judge directly on each held-out trace, `judge(trace=t)`. A direct call
returns a `Feedback` and logs nothing, so measuring agreement does not add verdicts to the traces. The
human side is each trace's latest `quality` assessment with source type `HUMAN`:

In [ ]:
for t in check[:5]:
    human = {a.source.source_id: a.value for a in t.info.assessments
             if a.name == "quality" and a.source.source_type == "HUMAN"}
    print(t.info.trace_id[:14], human, traces.question_of(t)[:50])

## 2. Align when it drifts

MLflow recommends at least ten labeled traces with at least 30% in each class. `judge.align()`
tunes the judge's instructions to the reviewers' labels (MemAlign with an embedding model,
SIMBA otherwise); the aligned judge is registered **only if** held-out agreement improves. The
nightly `align` job does exactly this, and stores the agreement on the experiment so triage can
discount judge-only failures when the judge is not trusted.

In [ ]:
if judge is not None:
    result = align_job.align(force=True)   # force: align even if agreement is already above the threshold
    print(result.metrics)

**Under the hood:** the `align` job is a run with `si.job=align`. It calls `judge.align(fit_traces,
optimizer=...)` with `MemAlignOptimizer(reflection_lm=..., embedding_model=...)` when an embedding model
is configured, `SIMBAAlignmentOptimizer(model=...)` otherwise. The optimizer learns from the HUMAN
`quality` assessments on those traces. Only if held-out agreement improves does it call
`aligned.register(name="quality")`, which stores the judge in the experiment's scorer registry. The
agreement goes into the experiment tag `si.judge_agreement`, which triage reads.

In [ ]:
from mlflow.genai.scorers import list_scorers

print("si.judge_agreement:", mlflow.get_experiment(experiment_id).tags.get("si.judge_agreement", "(not measured yet)"))
print("registered scorers:", [s.name for s in list_scorers()] or "(none)")
for r in mlflow.search_runs(filter_string="tags.`si.job` = 'align'", output_format="list"):
    print(r.info.run_name, r.data.tags.get("si.outcome"), r.data.metrics)

### Calibrate the first pass

A decision model has no rationale to align. What you tune is the **band** of pass probabilities
that gets re-judged, and you choose it from your own labels: where do Jev's disagreements with
people fall? (In MLflow's test on 72 harder answers, every miss scored between about 0.5 and 0.8.)

In [ ]:
if config.settings.has_fast_judge:
    cal = align_job.calibrate_first_pass(check)
    print(f"Jev agrees with people on {cal['fast_agreement']:.0%} of {cal['labels']} held-out labels")
    print(f"{cal['fast_misses_in_band']} of {cal['fast_misses']} misses fall in the band {config.settings.fast_judge_band}, "
          f"which re-judges {cal['fast_rejudge_share']:.0%} of verdicts")
    print("narrowest band containing every miss:", cal["suggested_band"], "-> SI_FAST_JUDGE_BAND")
else:
    print("No first-pass model configured.")

## 3. Automatic evaluation, for when traffic is real

Reviewer labels arrive unevenly. Registered judges can sample live traces on the server instead:

```python
from mlflow.genai.scorers import ScorerSamplingConfig
signals.quality_judge().register(name="quality").start(
    sampling_config=ScorerSamplingConfig(sample_rate=0.1, filter_string="tags.si_source = 'production'"))
```

(`selfimprove online`. It needs `mlflow server` with a database backend, so it is not run here.)
With a first-pass model, `online` samples Jev at 100% (`--fast-sample-rate`) into
`quality_first_pass`, and `selfimprove score` turns each of those into a `quality` verdict: copied
when Jev is confident, re-judged with a rationale when it is not. Code checks and span checks
cannot be registered on open-source MLflow, so `score` runs them.

## 4. An optimizer writes a candidate

`optimize_prompts` takes the instructions, the **train** rows and the scorers, and returns a new
version. `GepaPromptOptimizer` evolves variants by reflecting on scorer rationales (thorough,
expensive); `MetaPromptOptimizer` restructures the prompt with a metaprompt (cheap, suits little
data). The predict function loads the prompt inside each call, which is how MLflow swaps each
candidate in.

In [ ]:
from selfimprove import improvers

OPTIMIZER = "metaprompt"   # or "gepa" (budget ~60 metric calls: a few dollars on the default models)

train = store.rows("train")
print(len(train), "train rows; the optimizer never sees the", len(store.rows("holdout")), "holdout rows")
if config.settings.has_llm:
    text, why = improvers.optimize(train, signals.live() + signals.reference(), kind=OPTIMIZER, budget=30)
    candidate = instructions.register_candidate(text, f"{OPTIMIZER}: {why}", author=f"improver:{OPTIMIZER}")
    print(why)
    print(instructions.diff("production", candidate))

**Under the hood:** `improvers.optimize()` calls `mlflow.genai.optimize_prompts(predict_fn=
agents.predict_fn(version), train_data=<train rows>, prompt_uris=[<production prompt uri>],
optimizer=MetaPromptOptimizer(reflection_model=...)` (or `GepaPromptOptimizer(reflection_model=...,
max_metric_calls=budget)`), `scorers=..., aggregation=mean_of_scores)`. While it runs, MLflow swaps each
candidate text into the `load_prompt()` call inside `agents.run()`, so the agent code does not change. The best
text comes back as `result.optimized_prompts[0]`, and `instructions.register_candidate()` registers it as
a new prompt version tagged `author` and `status=candidate`. No alias moves.

In [ ]:
for v in mlflow.MlflowClient().search_prompt_versions(config.settings.prompt_name):
    print(f"v{v.version}  author={v.tags.get('author', '-'):<20} status={v.tags.get('status')}")

**Read the diff.** Optimizers can win by pleasing the judge: longer, more hedged, judge-flattering
text. Watch for facts pasted in instead of a pointer to the source, and for instructions about the
evaluation itself.

## 5. Gate on the holdout

An optimizer raises the training score by design, so the holdout margin is the check that matters.
A candidate that gains on train but not on the holdout has probably overfit the suite.

In [ ]:
if config.settings.has_llm:
    decision = gate.compare(candidate, "production")
    print(decision.table())
    print(decision.summary())

**Under the hood:** the same `gate.compare()` as notebook 03: a parent run tagged `si.job=gate`,
four nested `evaluate()` runs, and the table saved as the `gate_table.txt` artifact. Any gate decision
can be read back later from MLflow alone. Without credentials this shows the gate run from
notebook 03.

In [ ]:
last = mlflow.search_runs(filter_string="tags.`si.job` = 'gate'", order_by=["start_time DESC"],
                         max_results=1, output_format="list")
if last:
    r = last[0]
    print(r.info.run_name, "| si.gate_passed =", r.data.tags.get("si.gate_passed"))
    print(mlflow.artifacts.load_text(f"runs:/{r.info.run_id}/gate_table.txt"))

Whether to promote is still your call here (`instructions.promote(candidate, ...)`). Notebook 06
lets the gate make it.

**Next:** [06 · The loop runs itself](06_automate.ipynb).